# EDA - Análise Exploratória de Dados
## Previsão de Churn - Telco Customer

**Objetivo:** explorar o dataset, entender padrões nos dados e preparar para modelagem.

**Dataset:** Telco Customer Churn (IBM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

print('Bibliotecas carregadas com sucesso!')

## 1. Carregar os Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')

print(f'Dataset carregado!')
print(f'Shape: {df.shape[0]} linhas x {df.shape[1]} colunas')
print()
df.head(10)

## 2. Estrutura dos Dados

In [ ]:
# Informacoes gerais do dataset
df.info()

In [ ]:
# Estatisticas descritivas - variaveis numericas
df.describe()

In [ ]:
# Estatisticas descritivas - variaveis categoricas
df.describe(include='object')

## 3. Qualidade dos Dados
### 3.1 Missing Values

In [ ]:
# Verificar missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'missing': missing,
    'percentual (%)': missing_pct
}).sort_values('percentual (%)', ascending=False)

print('Missing values por coluna:')
print(missing_df[missing_df['missing'] > 0])

if missing_df['missing'].sum() == 0:
    print('Nenhum missing value encontrado (mas vamos verificar valores vazios).')

# Verificar strings vazias em TotalCharges (problema comum neste dataset)
empty_total = (df['TotalCharges'] == ' ').sum()
print(f'\nTotalCharges com valores vazios (string vazia): {empty_total}')

### 3.2 Duplicados

In [ ]:
# Verificar duplicatas
duplicados = df.duplicated().sum()
print(f'Registros duplicados: {duplicados}')

### 3.3 Correções de Tipo

In [ ]:
# Corrigir TotalCharges: converter para numerico (strings vazias viram NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificar quantos NaN foram criados
nans_total = df['TotalCharges'].isna().sum()
print(f'TotalCharges - NaN apos conversao: {nans_total}')

# Preencher NaN com mediana
mediana = df['TotalCharges'].median()
df['TotalCharges'].fillna(mediana, inplace=True)
print(f'TotalCharges - NaN preenchidos com mediana ({mediana:.2f})')

# Converter SeniorCitizen de 0/1 para No/Yes (para facilitar analise)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
print('SeniorCitizen convertido para No/Yes')

# Remover customerID (nao e feature util)
df.drop('customerID', axis=1, inplace=True)
print('Coluna customerID removida')

print(f'\nShape final: {df.shape}')
df.dtypes

## 4. Análise do Target (Churn)

In [ ]:
# Distribuicao do target
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Distribuicao do Churn:')
print(churn_counts)
print()
print('Percentual:')
print(churn_pct.round(2))

# Grafico
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Countplot
sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Distribuicao do Churn', fontsize=14)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporcao do Churn', fontsize=14)

plt.tight_layout()
plt.show()

## 5. Análise das Variáveis Categóricas

In [ ]:
# Variaveis categoricas para analise
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaperlessBilling', 'PaymentMethod']

n_cols = 4
n_rows = (len(cat_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='Churn',
                 palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'Churn por {col}', fontsize=11)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='Churn', fontsize=8)

# Remover subplots vazios
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 6. Análise das Variáveis Numéricas

In [ ]:
# Histogramas das variaveis numericas por Churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='Churn', kde=True,
                palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'Distribuicao de {col} por Churn', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots das variaveis numericas por Churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='Churn', y=col,
               palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'{col} por Churn', fontsize=12)

plt.tight_layout()
plt.show()

## 7. Correlações

In [ ]:
# Criar copia com Churn numerico para correlacao
df_corr = df.copy()
df_corr['Churn_numeric'] = (df_corr['Churn'] == 'Yes').astype(int)

# Selecionar colunas numericas
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_numeric']

# Matriz de correlacao
corr_matrix = df_corr[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax)
ax.set_title('Matriz de Correlacao', fontsize=14)
plt.tight_layout()
plt.show()

print('\nCorrelacao com Churn:')
print(corr_matrix['Churn_numeric'].sort_values(ascending=False))

## 8. Principais Insights da EDA

Com base na análise exploratória, os principais insights são:

- **Desbalanceamento moderado:** ~26% dos clientes cancelaram (Churn=Yes), ~74% não cancelaram
- **Tipo de contrato:** clientes com contrato **mensal** têm taxa de churn muito maior que contratos anuais ou bienais
- **Tenure (tempo de casa):** clientes mais **novos** (tenure baixo) cancelam significativamente mais
- **Cobrança mensal:** clientes com **MonthlyCharges mais altos** tendem a cancelar mais
- **Serviços de segurança:** clientes **sem OnlineSecurity e TechSupport** cancelam mais
- **Método de pagamento:** **Electronic check** tem a maior taxa de churn
- **Internet por fibra ótica:** clientes com **Fiber optic** cancelam mais do que DSL ou sem internet
- **Gênero:** não parece ter impacto significativo no churn

Esses padrões serão utilizados nas próximas etapas de Feature Engineering e modelagem.

## 9. Salvar Dados Processados

In [ ]:
# Salvar dados limpos
output_path = '../data/processed/telco_churn_clean.csv'
df.to_csv(output_path, index=False)

print(f'Dados processados salvos em: {output_path}')
print(f'Shape: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Colunas: {list(df.columns)}')

## Próximos Passos

1. **Feature Engineering** — criar novas variáveis derivadas
2. **Baselines** — treinar DummyClassifier + Logistic Regression + Random Forest
3. **MLflow** — registrar experimentos e métricas
4. **MLP PyTorch** — treinar rede neural e comparar com baselines